# Modélisation baseline

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
# Chargement des splits
df_train = pd.read_csv(r"..\data_finale\train.csv", encoding="utf-8-sig")
df_val   = pd.read_csv(r"..\data_finale\val.csv", encoding="utf-8-sig")

In [12]:
# Sélection de la cible (Y) et des 2 variables naïves (X)
X_cols_naives = ["Performance_Gls", "Playing Time_MP"]
target_col = "market_value_in_eur"

In [13]:
# Correction des NA
# On filtre les dataframes pour ne garder que les lignes qui ont toutes leurs données
# sur les variables prédictives (X) ET la cible (y).
cols_a_verifier = X_cols_naives + [target_col]

df_train_clean = df_train.dropna(subset=cols_a_verifier)
df_val_clean = df_val.dropna(subset=cols_a_verifier)

print(f"Lignes après suppression des NA -> Train: {len(df_train_clean)} (vs {len(df_train)}) | Val: {len(df_val_clean)} (vs {len(df_val)})\n")

Lignes après suppression des NA -> Train: 7419 (vs 8149) | Val: 2425 (vs 2649)



In [14]:
# On extrait du jeu d'entraînement uniquement les colonnes de performance choisies
X_train = df_train_clean[X_cols_naives]
# On extrait la colonne que le modèle doit apprendre à prédire (la valeur marchande réelle)
y_train = df_train_clean[target_col]

# On effectue exactement la même séparation sur le jeu de validation
X_val = df_val_clean[X_cols_naives]
y_val = df_val_clean[target_col]

print(f"Variables utilisées pour le modèle naïf : {X_cols_naives}")
print(f"Variable cible : {target_col}\n")

Variables utilisées pour le modèle naïf : ['Performance_Gls', 'Playing Time_MP']
Variable cible : market_value_in_eur



In [15]:
# Entraînement du modèle naïf (Régression Linéaire)
modele_naif = LinearRegression()
modele_naif.fit(X_train, y_train)

# Prédictions sur le jeu d'entraînement et de validation
y_pred_train = modele_naif.predict(X_train)
y_pred_val = modele_naif.predict(X_val)

In [16]:
# Calcul des métriques de performance
def calculer_metriques(y_reel, y_pred):
    mae = mean_absolute_error(y_reel, y_pred)
    rmse = np.sqrt(mean_squared_error(y_reel, y_pred))
    r2 = r2_score(y_reel, y_pred)
    return mae, rmse, r2

mae_train, rmse_train, r2_train = calculer_metriques(y_train, y_pred_train)
mae_val, rmse_val, r2_val = calculer_metriques(y_val, y_pred_val)

In [17]:
# Affichage des résultats
print("Performances du modèle naïf (baseline)")
print()
print(f"Jeu d'entraînement (train - saisons 2020-2023)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_train:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_train:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_train:.4f} ({r2_train*100:.1f}%)")
print()
print(f"Jeu de validation (val - saison 2024)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_val:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_val:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_val:.4f} ({r2_val*100:.1f}%)")

# Un petit aperçu visuel des erreurs
df_comparaison = pd.DataFrame({
    "Joueur": df_val_clean["player"],
    "Saison": df_val_clean["season_year"],
    "Valeur Réelle": y_val,
    "Prédiction Naïve": y_pred_val,
    "Erreur (Ecart)": np.abs(y_val - y_pred_val)
})

print("\nExemple de prédictions du modèle naïf sur le jeu de validation :")
display(df_comparaison.sort_values(by="Erreur (Ecart)", ascending=False).head(5))

Performances du modèle naïf (baseline)

Jeu d'entraînement (train - saisons 2020-2023)
- MAE  (Erreur Moyenne Absolue) : 8,399,367.31 €
- RMSE (Écart-type des erreurs) : 13,386,005.65 €
- R²   (Pouvoir explicatif)     : 0.2577 (25.8%)

Jeu de validation (val - saison 2024)
- MAE  (Erreur Moyenne Absolue) : 9,167,100.06 €
- RMSE (Écart-type des erreurs) : 15,621,504.12 €
- R²   (Pouvoir explicatif)     : 0.2549 (25.5%)

Exemple de prédictions du modèle naïf sur le jeu de validation :


,Joueur,Saison,Valeur Réelle,Prédiction Naïve,Erreur (Ecart)
2511,Vinicius Júnior,2023,180000000.0,3.539053e+07,1.446095e+08
1281,Jude Bellingham,2023,180000000.0,4.313438e+07,1.368656e+08
1407,Kylian Mbappé,2023,180000000.0,5.781020e+07,1.221898e+08
719,Erling Haaland,2023,180000000.0,5.835145e+07,1.216486e+08
771,Federico Valverde,2023,120000000.0,1.495897e+07,1.050410e+08
